# Behind the scenes: Day Three
### SDM dataset preparation

**Prerequisites:** run `day2_bts.ipynb` first, so that `data/processed/day2/gbif_mammals_philippines.csv`
and `data/processed/kg_philippines_present.nc` exist.

This notebook:
1. Downloads WorldClim BIO5 & BIO14 (30 arc-sec, 1970–2000), Copernicus DEM GLO-30, and Hansen Global
   Forest Change (national, extended from Day 2's Negros-only pull) — all aligned to the same 1 km
   national grid as the existing KG classification.
2. Cleans Philippine tarsier (*Carlito syrichta*) occurrence records, generates pseudo-absences
   (excluding the bbox's Borneo corner), and extracts `kg_class`/`bio5`/`bio14`/`elevation`/
   `treecover2000` at every point into an SDM-ready table.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

from workshop_utils import RAW_DIR, PROCESSED_DIR
from workshop_utils.download import download_worldclim_bioclim, build_dem_mosaic, build_hansen_mosaic
print('Ready.')

---
## 1. WorldClim 2.1 — BIO5 & BIO14 (30 arc-sec, 1970–2000 baseline)

All three national-scale environmental layers should share one 1 km grid: the existing KG
classification (`kg_philippines_present.nc`) and land-sea mask, plus these two WorldClim bioclim
variables. 30 arc-sec (~1 km) is only offered for WorldClim's 1970–2000 historical-climate baseline —
the "2000–present" monthly weather product tops out at 2.5 arcmin (~4.5 km), so this is a hard
resolution/period trade, not a preference. Conceptually that's fine: bioclim variables represent a
long-term climate normal for a species' niche (the same role the existing ERA5-derived
"1991–2020 mean" climatology plays elsewhere in the workshop) — just a different 30-year window.

- **BIO5** — Max Temperature of Warmest Month
- **BIO14** — Precipitation of Driest Month

These two were picked because a Maxent SDM study on two other Philippine endemics
(*Varanus palawanensis*, *Caprimulgus manillensis*) found precipitation-of-driest-month and
temperature-of-warmest-month/quarter were consistently the most informative predictors for both
species (see `example_study.pdf`).

WorldClim distributes bioclim data as one 9.7 GB zip per resolution containing all 19 variables as
separate GeoTIFFs. `geodata.ucdavis.edu` supports HTTP range requests, so GDAL's
`/vsizip/vsicurl/` virtual filesystem lets us crop straight to the Philippines bbox without ever
downloading the full archive — the same "only fetch the window you need from a remote raster"
pattern already used for Hansen forest-cover data in `day2_bts.ipynb`.

In [ ]:
PH_BBOX = (113.5, 4.5, 127.0, 21.5)  # W, S, E, N -- matches kg_philippines_present.nc / land-sea mask extent

kg_grid = xr.open_dataset(PROCESSED_DIR / 'kg_philippines_present.nc')
PH_LAT, PH_LON = kg_grid['lat'].values, kg_grid['lon'].values
print(f'Target grid (from kg_philippines_present.nc): {len(PH_LAT)} x {len(PH_LON)} @ ~30 arc-sec')

BIOCLIM_VARS = {
    'bio5':  {'num': 5,  'long_name': 'Max Temperature of Warmest Month', 'units': 'degC'},
    'bio14': {'num': 14, 'long_name': 'Precipitation of Driest Month',    'units': 'mm'},
}

raw_arrays = {}
for name, meta in BIOCLIM_VARS.items():
    da = download_worldclim_bioclim(meta['num'], PH_BBOX)
    da.name = name
    da.attrs.update(long_name=meta['long_name'], units=meta['units'])
    raw_arrays[name] = da
    print(f"  {name}: shape={da.shape}, range=({float(da.min()):.1f}, {float(da.max()):.1f}) {meta['units']}")

raw_ds = xr.Dataset(raw_arrays)
raw_ds.attrs['description'] = 'WorldClim 2.1 bioclimatic variables, cropped to Philippines bbox, native 30 arc-sec grid (pre-alignment)'
raw_ds.attrs['source'] = 'WorldClim 2.1 historical climate (1970-2000 baseline), https://worldclim.org/data/worldclim21.html'

wc_raw_dir = RAW_DIR / 'WorldClim'
wc_raw_dir.mkdir(parents=True, exist_ok=True)
raw_out = wc_raw_dir / 'wc2.1_30s_bio5_bio14_philippines_raw.nc'
raw_ds.to_netcdf(raw_out)
print(f'saved {raw_out}')

In [ ]:
# Align onto the exact KG / land-sea-mask grid (nearest-neighbour -- the grids already almost
# coincide since we cropped to the same bbox/resolution, this just guarantees pixel-for-pixel
# alignment with the other national 1 km layers)
bio_ds = raw_ds.reindex(lat=PH_LAT, lon=PH_LON, method='nearest')
bio_ds.attrs['description'] = 'WorldClim 2.1 BIO5 (max temp of warmest month) and BIO14 (precip of driest month), aligned to the same 1 km grid as kg_philippines_present.nc'
bio_ds.attrs['period'] = '1970-2000 (WorldClim 2.1 historical climate baseline)'
bio_ds.attrs['source'] = 'WorldClim 2.1, https://worldclim.org/data/worldclim21.html'

bio_out = PROCESSED_DIR / 'worldclim_bio_philippines_1970_2000.nc'
bio_ds.to_netcdf(bio_out)
print(f'saved {bio_out}  ({bio_out.stat().st_size / 1e6:.1f} MB)')
bio_ds

---
## 2. Copernicus DEM GLO-30 — aggregated to the same 1 km grid

Elevation, via the same public STAC pattern already used for Landsat/Sentinel-2 in `day2_bts.ipynb`
(Microsoft Planetary Computer, `cop-dem-glo-30`, no credentials needed). Native resolution is ~30 m
(3600×3600 px per 1°×1° tile) — far too fine to carry as a national array (the full PH bbox would be
~2.5 billion pixels). Since both the DEM tiles and our target grid are plain lat/lon (EPSG:4326), no
reprojection is needed: each tile is read directly at a *decimated* resolution matching however many
target 1 km cells it covers, using block-averaging, and written straight into its footprint on the
shared national grid.

**Deliberately no raw copy of the native 30 m data is kept anywhere** — every tile is streamed via a
windowed remote read and only the aggregated 1 km result ever touches disk.

In [ ]:
dem = build_dem_mosaic(PH_BBOX, PH_LAT, PH_LON)
print(f'elevation range: ({np.nanmin(dem):.1f}, {np.nanmax(dem):.1f}) m, {np.isnan(dem).mean():.1%} NaN (open ocean outside all tiles)')

# sanity check against the land-sea mask: land pixels should have real elevation values
lsm = xr.open_dataset(PROCESSED_DIR / 'land-sea-mask_0p00833333.nc')['land_sea_mask'].values
land = lsm > 0.5
print(f'land pixels missing elevation: {np.isnan(dem[land]).sum()} / {land.sum()}')

In [ ]:
dem_ds = xr.Dataset(
    {'elevation': (('lat', 'lon'), dem)},
    coords={'lat': PH_LAT, 'lon': PH_LON},
)
dem_ds['elevation'].attrs.update(long_name='Elevation above EGM2008 geoid', units='m')
dem_ds.attrs['description'] = (
    'Copernicus DEM GLO-30, aggregated (block-averaged) from native ~30 m to the same 1 km grid as '
    'kg_philippines_present.nc. Native-resolution tiles are streamed via windowed remote reads and '
    'never stored locally.'
)
dem_ds.attrs['source'] = 'Copernicus DEM GLO-30, via Microsoft Planetary Computer STAC (cop-dem-glo-30)'

dem_out = PROCESSED_DIR / 'dem_philippines_1km.nc'
dem_ds.to_netcdf(dem_out)
print(f'saved {dem_out}  ({dem_out.stat().st_size / 1e6:.1f} MB)')
dem_ds

---
## 3. Hansen Global Forest Change — national, aggregated to the same 1 km grid

Canopy cover, extended from Day 2's Negros-only pull to the full national bbox. Forest structure is
very plausibly a bigger driver of tarsier presence than climate alone (dense understory for
concealment and insect abundance) — climate-only predictors couldn't recover the species' known
range in the first Part III pass, so this is the natural next predictor to add.

Hansen's public GeoTIFFs are row-strip organized (no internal tiling, no overviews) — unlike the
Copernicus DEM tiles above, a decimated read still has to fetch every full-width row in the requested
window, so this takes noticeably longer (roughly 1-2 minutes per tile with real overlap, 6 tiles
needed to cover the country) rather than DEM's ~20 seconds for the whole country. Still no native
30 m file ever touches disk — only the aggregated 1 km result.

In [ ]:
HANSEN_TILES = ['10N_110E', '10N_120E', '20N_110E', '20N_120E', '30N_110E', '30N_120E']

treecover = build_hansen_mosaic(PH_BBOX, HANSEN_TILES, PH_LAT, PH_LON)
print(f'treecover grid: {treecover.shape}, {np.isnan(treecover).mean():.1%} NaN')
print(f'treecover range: ({np.nanmin(treecover):.1f}, {np.nanmax(treecover):.1f}) %')

lsm_check = xr.open_dataset(PROCESSED_DIR / 'land-sea-mask_0p00833333.nc')['land_sea_mask'].values
land_check = lsm_check > 0.5
print(f'land pixels missing treecover: {np.isnan(treecover[land_check]).sum()} / {land_check.sum()}')

In [ ]:
forest_ds = xr.Dataset(
    {'treecover2000': (('lat', 'lon'), treecover)},
    coords={'lat': PH_LAT, 'lon': PH_LON},
)
forest_ds['treecover2000'].attrs.update(long_name='Percent canopy cover, year 2000', units='%')
forest_ds.attrs['description'] = (
    'Hansen Global Forest Change v1.11 treecover2000, aggregated (block-averaged) from native ~30 m '
    'to the same 1 km grid as kg_philippines_present.nc. Native-resolution tiles are streamed via '
    'windowed remote reads and never stored locally.'
)
forest_ds.attrs['source'] = 'Hansen et al. (2013), GFC v1.11, https://storage.googleapis.com/earthenginepartners-hansen/'

forest_out = PROCESSED_DIR / 'forest_cover_philippines_1km.nc'
forest_ds.to_netcdf(forest_out)
print(f'saved {forest_out}  ({forest_out.stat().st_size / 1e6:.1f} MB)')
forest_ds

---
## 4. Load and clean Philippine tarsier occurrence records

Unlike the flying fox/eagle in Day 2, the tarsier doesn't have its own dedicated pre-cleaned file --
pull it out of the broader `gbif_mammals_philippines.csv` compilation.

In [ ]:
mammals = pd.read_csv(PROCESSED_DIR / 'day2' / 'gbif_mammals_philippines.csv', low_memory=False)

tarsier_raw = mammals[mammals['species'] == 'Carlito syrichta'].dropna(
    subset=['decimalLongitude', 'decimalLatitude'])
tarsier_raw = tarsier_raw[
    tarsier_raw['decimalLongitude'].between(113.5, 127.0) &
    tarsier_raw['decimalLatitude'].between(4.5, 21.5)
]
print(f'Raw records: {len(tarsier_raw)}')

# Native range = the Mindanao faunal region (Mindanao, Bohol, Leyte, Samar, nearby islands). A
# handful of points fall well outside it (central Luzon, Mindoro/Romblon) -- almost certainly
# captive/zoo records mislabeled as wild. Drop them; keep the Bohol Sanctuary cluster (still real
# habitat -- flagged as a sampling-bias caveat for Part IV, not removed here).
out_of_range = (
    (tarsier_raw['decimalLatitude'] > 13.5) |
    ((tarsier_raw['decimalLongitude'] < 122.5) & (tarsier_raw['decimalLatitude'] > 11.5))
)
print(f'Dropping {int(out_of_range.sum())} likely captive/out-of-range records')
tarsier = tarsier_raw[~out_of_range].copy()

# Remove spatial duplicates rounded to 0.01° (~1 km)
tarsier['lon_r'] = tarsier['decimalLongitude'].round(2)
tarsier['lat_r'] = tarsier['decimalLatitude'].round(2)
tarsier = tarsier.drop_duplicates(subset=['lon_r', 'lat_r'])
tarsier = tarsier.rename(columns={'decimalLongitude': 'lon', 'decimalLatitude': 'lat'})[['lon', 'lat']]

print(f'Clean presence set: {len(tarsier)} points')
tarsier.head()

---
## 5. Generate pseudo-absence (background) points

Sample random points from Philippine land area, using the 1 km land-sea mask. The rectangular PH
bbox spills into northern Borneo (Sabah) -- confirmed earlier by the DEM's ~3900 m max elevation
matching Mt. Kinabalu, not any Philippine peak -- so that corner is excluded explicitly. The
land-sea mask alone won't catch this: it only knows land vs. ocean, not country.

In [ ]:
lsm = xr.open_dataset(PROCESSED_DIR / 'land-sea-mask_0p00833333.nc')
PH_LAT = lsm['lat'].values
PH_LON = lsm['lon'].values
land_mask = lsm['land_sea_mask'].values > 0.5

# Exclude the bbox's Borneo corner (roughly lon < 119, lat < 7.5) from the background pool
lon_grid, lat_grid = np.meshgrid(PH_LON, PH_LAT)
borneo_corner = (lon_grid < 119.0) & (lat_grid < 7.5)
land_mask = land_mask & ~borneo_corner

land_rows, land_cols = np.where(land_mask)
print(f'Land pixels available (Philippines only, Borneo corner excluded): {len(land_rows):,}')

n_pseudo = min(len(tarsier) * 10, 2000)   # up to 10x presence count, capped at 2000
rng = np.random.default_rng(42)
idx = rng.choice(len(land_rows), size=n_pseudo, replace=False)
pseudo_df = pd.DataFrame({
    'lon': PH_LON[land_cols[idx]],
    'lat': PH_LAT[land_rows[idx]],
})

print(f'Generated {len(pseudo_df)} pseudo-absence points')

---
## 6. Extract predictors at all points

`kg_philippines_present.nc`, `worldclim_bio_philippines_1970_2000.nc`, `dem_philippines_1km.nc`, and
`forest_cover_philippines_1km.nc` all share the exact same 1 km grid as the land-sea mask above -- so
the nearest-grid-cell lookup (the `np.argmin(np.abs(array - value))` trick from Day 2's KG work) only
needs to happen **once** per point, then gets reused across all five layers.

In [ ]:
kg_grid    = xr.open_dataset(PROCESSED_DIR / 'kg_philippines_present.nc')
bio_ds     = xr.open_dataset(PROCESSED_DIR / 'worldclim_bio_philippines_1970_2000.nc')
dem_ds     = xr.open_dataset(PROCESSED_DIR / 'dem_philippines_1km.nc')
forest_ds  = xr.open_dataset(PROCESSED_DIR / 'forest_cover_philippines_1km.nc')


def nearest_rowcol(lons, lats, grid_lon=PH_LON, grid_lat=PH_LAT):
    """Nearest-grid-cell index for every (lon, lat) point, vectorised over the whole grid at once."""
    col_idx = np.abs(grid_lon[None, :] - np.asarray(lons)[:, None]).argmin(axis=1)
    row_idx = np.abs(grid_lat[None, :] - np.asarray(lats)[:, None]).argmin(axis=1)
    return row_idx, col_idx


def extract_predictors(df):
    row_idx, col_idx = nearest_rowcol(df['lon'].values, df['lat'].values)
    out = df.copy()
    out['kg_class']      = kg_grid['kg_class'].values[row_idx, col_idx]
    out['bio5']          = bio_ds['bio5'].values[row_idx, col_idx]
    out['bio14']         = bio_ds['bio14'].values[row_idx, col_idx]
    out['elevation']     = dem_ds['elevation'].values[row_idx, col_idx]
    out['treecover2000'] = forest_ds['treecover2000'].values[row_idx, col_idx]
    return out


print('Extracting values at presence points ...')
tarsier_out = extract_predictors(tarsier)
tarsier_out['presence'] = 1

print('Extracting values at pseudo-absence points ...')
pseudo_out = extract_predictors(pseudo_df)
pseudo_out['presence'] = 0

sdm_df = pd.concat([tarsier_out, pseudo_out], ignore_index=True).dropna()
n_pres = int(sdm_df['presence'].sum())
n_abs  = int((sdm_df['presence'] == 0).sum())
print(f'\nSDM dataset: {n_pres} presences + {n_abs} background points')
sdm_df.describe()

---
## 7. Save processed dataset

This is a Day-3-specific derived output (not a reusable reference layer like KG/climate/DEM/WorldClim/
forest cover), so it goes under `data/processed/day3/` rather than the shared top level.

In [ ]:
day3_dir = PROCESSED_DIR / 'day3'
day3_dir.mkdir(parents=True, exist_ok=True)

out_path = day3_dir / 'sdm_tarsier.csv'
sdm_df.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
sdm_df.head(10)